# MiniReasoner

Train the 32M-parameter base model on a GPU VM, resume a training run, or generate text from a checkpoint. The notebook uses the SimpleStories training split with a 1% validation holdout. The published test split stays reserved for final evaluation.

## Open the VM project

From the MiniReasoner checkout, run `uv sync --locked --python 3.12`. Open this notebook in Jupyter or VS Code Remote SSH and select the project's `.venv` kernel. The first cell loads the project's `.env`.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

working_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / "src" / "train.py").is_file() and (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the MiniReasoner repository.")
load_dotenv(project_root / ".env")
print(f"Using MiniReasoner source in {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Run settings

The **base** preset trains on CUDA. The **small** preset supports both CUDA and CPU. To train locally on CPU, set `preset = "small"` and `training_device = "cpu"`. Automatic precision uses FP32 on CPU and T4 GPUs, and BF16 on GPUs with native support. For fresh data preparation, `tokenizer_device` independently selects CPU or CUDA for tokenizer training.

For DagsHub charts, set `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, and `MLFLOW_TRACKING_PASSWORD` in the project's `.env`. Use a DagsHub access token as the password. With a blank tracking URI, the notebook uses SQLite and backs up the database to Drive.

Checkpoints default to `gdrive:MiniReasoner/checkpoints/<run-id>/`. Authorize the VM's `gdrive` remote once with `uv run rclone config`, or set `MINIREASONER_CHECKPOINT_ROOT` to a writable Drive mount. For the SQLite dashboard in the JarvisLabs browser, set `MINIREASONER_JARVIS_ORIGIN` to the notebook's HTTPS origin.

To download your prepared data on a GPU VM, set `MINIREASONER_CORPUS_DRIVE_URL` in the project's `.env` to the Google Drive folder link for `full-val-0.01`, or paste the link into `corpus_drive_url` below. The folder must allow anyone with the link to view and download its files. The notebook checks the local corpus first.

The base preset schedules one epoch and pauses at optimizer step **12,000** by default. Set **stop_after** to a later step or **None** to continue to the end of the same schedule. **CHECKPOINT_LIMIT = None** in `src/settings.py` keeps every completed checkpoint on Drive.

In [ ]:
import json
import math
from dataclasses import asdict
from hashlib import blake2b
from tempfile import TemporaryDirectory
from time import perf_counter

import numpy as np
import torch
from tokenizers import Tokenizer

from src.generate import complete
from src.model import GPT, GPTConfig
from src.settings import (
    DATASET_CACHE_DIR, MLFLOW_DB_SUBPATH, MLFLOW_EXPERIMENT, PRESETS,
    RCLONE_CHECKPOINT_ROOT, SIMPLESTORIES_TRAIN_FILES, TOKENIZED_DATA_DIR,
    VALIDATION_FRACTION, VOCAB_SIZE,
)
from src.tokenizer import (
    EOS, OUTPUT_FILES, TOKEN_DTYPE, pack_documents, prepared_corpus_manifest,
    publish_prepared_corpus, sha256, story_texts, train_tokenizer,
    verify_corpus_files, verify_packed_eos,
)
from src.checkpoint_store import CheckpointStore, check_checkpoint_root
from src.drive_tracking import DriveTracking
from src.metrics import track_run
from src.tracking import checkpoint_uploads, download_checkpoint
from src.train import (
    EpochBatchSampler, TokenStream, TrainConfig, evaluate, load_checkpoint, loss_for,
    resolve_training_device, resolve_precision, sample_validation_windows,
)

# Settings
preset = "base"
train_model = True
training_device = "cuda"  # The small preset also supports "cpu".
inference_device = "cpu"
precision_request = "auto"
tokenizer_device = "auto"  # Fresh tokenizer only. Choose "auto", "cpu", or "cuda".
corpus_drive_url = os.getenv("MINIREASONER_CORPUS_DRIVE_URL", "").strip()
stop_after = 12_000 if preset == "base" else None  # Absolute optimizer step.
resume = False
mlflow_run_id = None  # Leave empty for a fresh run. Paste a saved MLflow run ID to resume or generate.
checkpoint_root = os.getenv("MINIREASONER_CHECKPOINT_ROOT") or RCLONE_CHECKPOINT_ROOT
loss_decimals = 4

seed = 42
dataset_name = "SimpleStories/SimpleStories"
source_splits = {"train": "train", "val": "train"}

check_checkpoint_root(checkpoint_root)
tracking_backup = None
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "").strip() if train_model else None
if train_model and not tracking_uri:
    tracking_db = project_root / MLFLOW_DB_SUBPATH
    tracking_db.parent.mkdir(parents=True, exist_ok=True)
    tracking_backup = DriveTracking(tracking_db, checkpoint_root)
    tracking_backup.prepare()
    tracking_uri = f"sqlite:///{tracking_db.as_posix()}"

if (resume or not train_model) and not mlflow_run_id:
    raise ValueError("Set mlflow_run_id to the saved run ID to resume or generate.")

cache_dir = DATASET_CACHE_DIR
tokenized_path = TOKENIZED_DATA_DIR

## Download and inspect SimpleStories

Split the full published training split by a seeded hash of each cleaned story. The default is approximately 99% training and 1% validation. Identical stories stay in the same split. Empty stories and stories containing the reserved EOS marker are skipped. The notebook first checks for a matching packed corpus on the VM, then tries the configured Drive folder. If neither is available, it prepares the corpus from SimpleStories.

In [ ]:
if train_model:
    if not 0 < VALIDATION_FRACTION < 1:
        raise ValueError("VALIDATION_FRACTION must be between 0 and 1.")

    corpus_settings = {
        "dataset": dataset_name,
        "source_splits": source_splits,
        "seed": seed,
        "split_method": "story_hash_v1",
        "validation_fraction": VALIDATION_FRACTION,
        "target_vocab_size": VOCAB_SIZE,
    }

    manifest = prepared_corpus_manifest(tokenized_path, corpus_settings)

    if manifest is None and corpus_drive_url:
        tokenized_path.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(prefix=".import-", dir=tokenized_path.parent) as temporary:
            staging = Path(temporary) / tokenized_path.name
            staging.mkdir()
            import gdown

            print("Downloading prepared data from Google Drive...", flush=True)
            entries = gdown.download_folder(
                url=corpus_drive_url, skip_download=True, quiet=True, use_cookies=False, timeout=60,
            )
            files = [entry for entry in entries if entry.path in OUTPUT_FILES]
            if sorted(entry.path for entry in files) != sorted(OUTPUT_FILES):
                raise ValueError(f"Choose the Drive folder containing one copy of each file: {', '.join(OUTPUT_FILES)}.")
            for entry in files:
                gdown.download(id=entry.id, output=str(staging / entry.path),
                               use_cookies=False, timeout=60, retries=3)

            imported_manifest = prepared_corpus_manifest(staging, corpus_settings)
            if imported_manifest is None:
                raise ValueError("The prepared corpus is missing from the transfer.")
            verify_corpus_files(staging, imported_manifest)
            publish_prepared_corpus(staging, tokenized_path)
            manifest = imported_manifest

    reuse_corpus = manifest is not None
    if reuse_corpus:
        print(f"Found matching packed data in {tokenized_path}. File integrity is checked before training.")

In [ ]:
if train_model and not reuse_corpus:
    if not 0 < VALIDATION_FRACTION < 1:
        raise ValueError("VALIDATION_FRACTION must be between 0 and 1.")

    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
    from datasets import load_dataset

    token = os.getenv("HF_TOKEN", "").strip() or None
    print("Downloading SimpleStories with Hugging Face authentication." if token else
          "Downloading public SimpleStories anonymously. Set HF_TOKEN for authenticated access.")

    dataset = load_dataset(
        dataset_name,
        data_files={"train": SIMPLESTORIES_TRAIN_FILES},
        split="train",
        cache_dir=str(cache_dir),
        # The Hub metadata describes both splits, but only training shards were requested.
        verification_mode="no_checks",
        token=token,
    ).select_columns(["story"])
    del token

    # The same cleaned text always lands in the same split, including exact duplicates.
    split_indices = {"train": [], "val": []}
    validation_threshold = int(VALIDATION_FRACTION * (1 << 64))
    hash_key = str(seed).encode("ascii")
    row_index = 0
    for batch in dataset.iter(batch_size=2048):
        for text in batch["story"]:
            story = (text or "").strip()
            if story and EOS not in story:
                digest = blake2b(story.encode("utf-8"), digest_size=8, key=hash_key).digest()
                split_name = "val" if int.from_bytes(digest, "big") < validation_threshold else "train"
                split_indices[split_name].append(row_index)
            row_index += 1

    splits = {}
    for name, indices in split_indices.items():
        if not indices:
            raise ValueError(f"No usable {name} stories. Adjust VALIDATION_FRACTION.")
        splits[name] = dataset.select(indices)
    del split_indices, indices
    usable_count = sum(len(rows) for rows in splits.values())
    print(f"Used {usable_count:,} of {len(dataset):,} published training rows.")
    print(f"Actual split: {len(splits['train']) / usable_count:.2%} train / "
          f"{len(splits['val']) / usable_count:.2%} validation.")


    for name, rows in splits.items():
        sample_count = min(2048, len(rows))
        positions = np.linspace(0, len(rows) - 1, num=sample_count, dtype=np.int64)
        word_counts = [len(story.split()) for story in story_texts(rows.select(positions))]
        p10, median, p90 = np.percentile(word_counts, [10, 50, 90])
        print(f"{name}: {len(rows):,} stories")
        print(f"  Words per story (sample of {sample_count:,}): p10={p10:.0f}, median={median:.0f}, p90={p90:.0f}")

    example_story = splits["train"][0]["story"].strip()
    print("\nExample training story\n")
    print(example_story)

## Train the tokenizer and verify the packed data

Train byte-level BPE on the full local training portion and pack both splits in bounded batches. Before publishing the corpus, check that every selected story was packed, every recorded story boundary contains EOS, and each file has exactly one EOS per story. Only then write the completed manifest and publish the whole prepared directory. Reused corpora undergo file-size and checksum checks before training.

In [ ]:
if train_model and not reuse_corpus:
    fit_started = perf_counter()
    tokenizer = train_tokenizer(story_texts(splits["train"]), VOCAB_SIZE, device=tokenizer_device)
    print(f"Tokenizer training finished in {(perf_counter() - fit_started) / 60:.1f} minutes.")
    preview = tokenizer.encode(example_story, add_special_tokens=False)

    print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")
    print(f"First 25 tokens: {preview.tokens[:25]}")

In [ ]:
if train_model and not reuse_corpus:
    tokenized_path.parent.mkdir(parents=True, exist_ok=True)
    if tokenized_path.exists() and (not tokenized_path.is_dir() or any(tokenized_path.iterdir())):
        raise FileExistsError(f"The corpus destination contains existing files: {tokenized_path}")

    with TemporaryDirectory(prefix=".prepare-", dir=tokenized_path.parent) as temporary:
        staging = Path(temporary) / tokenized_path.name
        staging.mkdir()
        tokenizer_path = staging / "tokenizer.json"
        tokenizer_path.write_text(tokenizer.to_str(pretty=True), encoding="utf-8")
        saved_tokenizer = Tokenizer.from_file(str(tokenizer_path))
        eos_id = saved_tokenizer.token_to_id(EOS)
        if eos_id is None or eos_id != tokenizer.token_to_id(EOS):
            raise ValueError("The saved tokenizer has an invalid EOS token.")

        manifest = {
            "version": 1,
            "dtype": TOKEN_DTYPE.str,
            **corpus_settings,
            "source_rows": len(dataset),
            "vocab_size": saved_tokenizer.get_vocab_size(),
            "eos_id": eos_id,
            "tokenizer_sha256": sha256(tokenizer_path),
        }
        for name, rows in splits.items():
            print(f"Packing {len(rows):,} {name} stories...", flush=True)
            split_info, story_ends = pack_documents(
                staging / f"{name}.bin", saved_tokenizer, story_texts(rows)
            )
            if split_info["documents"] != len(rows):
                raise ValueError(
                    f"{name}: packed {split_info['documents']:,} stories, expected {len(rows):,}."
                )
            verify_packed_eos(staging / f"{name}.bin", split_info, story_ends, eos_id)
            manifest[name] = split_info
            print(f"{name}: verified all {len(rows):,} stories and their EOS boundaries.", flush=True)
            del story_ends

        (staging / "manifest.json").write_text(
            json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
        )
        publish_prepared_corpus(staging, tokenized_path)

    print(f"Completed tokenization and saved the verified corpus to {tokenized_path}")

In [ ]:
if train_model:
    for name in ("train", "val"):
        print(f"{name}: {manifest[name]['documents']:,} stories, {manifest[name]['tokens']:,} packed tokens")
    if not reuse_corpus:
        print("Both splits passed the full story-count and EOS-boundary checks before publication.")
    else:
        print("The saved corpus will pass file-size and checksum checks before training starts.")

## Load and check the packed corpus

Check the file hashes and split provenance before opening the training and validation streams. The model uses the vocabulary size recorded in the manifest.

In [ ]:
if train_model:
    settings = PRESETS[preset]
    manifest_path = tokenized_path / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Prepare the full SimpleStories corpus first: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("split_method") != "story_hash_v1":
        raise ValueError("Prepare the full SimpleStories split with the updated notebook.")
    if manifest["version"] != 1 or manifest["dtype"] != TOKEN_DTYPE.str:
        raise ValueError("Unsupported dataset format")
    if manifest.get("dataset") == "SimpleStories/SimpleStories" and manifest.get("source_splits") != {
        "train": "train",
        "val": "train",
    }:
        raise ValueError("Prepare SimpleStories with the current notebook so both splits come from the published training data")

    tokenizer_path = tokenized_path / "tokenizer.json"
    verify_corpus_files(tokenized_path, manifest)
    tokenizer_json = tokenizer_path.read_text(encoding="utf-8")
    vocab_size = manifest["vocab_size"]
    model_config = GPTConfig(vocab_size=vocab_size, **settings["model"])
    print(f"Model: n_embd={model_config.n_embd:,}, n_head={model_config.n_head}, "
          f"head_dim={model_config.n_embd // model_config.n_head}")
    streams = {}
    for split in ("train", "val"):
        path = tokenized_path / f"{split}.bin"
        size = manifest[split]["tokens"]
        streams[split] = TokenStream(path, size, model_config.context_length)
    print("Corpus verified: tokenizer and both packed splits match the saved manifest.")

    training_settings = settings["training"].copy()
    epochs = training_settings.pop("epochs", None)
    sampler = EpochBatchSampler(streams["train"], training_settings["batch_size"], training_settings["seed"])
    if epochs is not None:
        if not isinstance(epochs, int) or epochs < 1:
            raise ValueError("epochs must be a positive integer")
        training_settings["steps"] = epochs * sampler.steps_per_epoch
    config = TrainConfig(**training_settings)
    val_data = streams["val"]
    val_targets_per_check = math.ceil(val_data.target_count * config.eval_token_fraction)
    stop = config.steps if stop_after is None else stop_after
    if not 1 <= stop <= config.steps:
        raise ValueError("stop_after must be between 1 and the configured total steps")

    for split in ("train", "val"):
        print(f"{split}: {manifest[split]['documents']:,} stories, {manifest[split]['tokens']:,} tokens")

## Batch and epoch analysis

One epoch visits all training prediction targets once, using shuffled non-overlapping windows. The last window can be shorter. Each validation check shuffles integer IDs for non-overlapping windows in the packed validation tokens and scores half of the next-token targets by default. Samples can overlap across checks. The seed and optimizer step reproduce each selection on resume.

In [ ]:
if train_model:
    train_data = streams["train"]
    context_length = model_config.context_length
    batch_size = config.batch_size
    scheduled_targets = sampler.targets_through(stop)
    print(f"{preset.title()} preset: up to {batch_size:,} windows per batch, context {context_length:,}")
    print(f"Training: {train_data.target_count:,} targets in {sampler.window_count:,} windows")
    print(f"  One epoch: {sampler.steps_per_epoch:,} optimizer steps")
    print(f"  Through step {stop:,}: {scheduled_targets:,} targets "
          f"({scheduled_targets / train_data.target_count:.4f} epochs)")
    if train_data.tail_targets:
        print(f"  The last window contributes {train_data.tail_targets:,} targets.")

    sample_starts, _ = sample_validation_windows(val_data, config, step=0)
    initial_batches = math.ceil(len(sample_starts) / batch_size)
    fresh_run_checks = 1 + math.ceil(stop / config.eval_every)
    print(f"Validation per check: {val_targets_per_check:,} of {val_data.target_count:,} next-token targets "
          f"({config.eval_token_fraction:.1%}, rounded up)")
    print(f"  Step 0: {len(sample_starts):,} random windows in {initial_batches:,} batches of up to {batch_size:,}.")
    print(f"  Fresh-run schedule through step {stop:,}: {fresh_run_checks:,} checks, including step 0")

## Initialize or resume training

Leave **resume = False** and **mlflow_run_id = None** for a fresh run. To resume a MiniReasoner MLflow run, set **resume = True** and paste its printed run ID. Use the same tracking backend, Drive checkpoint root, training settings and packed corpus. The notebook restores the latest saved model and optimizer from Drive and reconstructs batch order from the saved step and seed.

In [ ]:
if train_model:
    device = resolve_training_device(training_device, preset)
    precision = resolve_precision(precision_request, device)
    device_name = torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU"
    print(f"Runtime: {device_name} | {precision}")

    torch.manual_seed(config.seed)
    model = GPT(model_config).to(device)
    decay, no_decay = [], []

    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)

    optimizer = torch.optim.AdamW(
        [{"params": decay}, {"params": no_decay, "weight_decay": 0.0}],
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95),
        fused=device.type == "cuda",
    )
    run_identity = {
        "model_config": asdict(model_config),
        "train_config": asdict(config),
        "sampling": "shuffled_windows_v1",
        "data_manifest": manifest,
    }
    start = 0
    if resume:
        with download_checkpoint(CheckpointStore(checkpoint_root, mlflow_run_id)) as checkpoint_path:
            checkpoint = load_checkpoint(checkpoint_path)
        if checkpoint["version"] != 1 or any(checkpoint.get(k) != v for k, v in run_identity.items()):
            raise ValueError(
                "Resume requires matching training settings and data. "
                "For a fresh run, set resume=False."
            )
        start = checkpoint["step"]
        if stop <= start:
            raise ValueError(
                f"stop_after must exceed the completed step ({start})")
        model.load_state_dict(checkpoint["model"])
        # Restore parameter groups using the optimizer implementation for this device.
        for group in checkpoint["optimizer"]["param_groups"]:
            group["fused"] = device.type == "cuda"
        optimizer.load_state_dict(checkpoint["optimizer"])
        history = checkpoint["metrics"]
        del checkpoint
    else:
        history = []

    training_started = False
    num_parameters = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"{num_parameters:,} parameters | {device} {precision} | "
        f"{sampler.steps_per_epoch:,} steps/epoch | steps {start + 1:,} to {stop:,}",
        flush=True,
    )

In [ ]:
# Open the link in a new tab to follow the training run.
if train_model:
    from src.mlflow_ui import open_mlflow_dashboard
    open_mlflow_dashboard(tracking_uri)

## Train and validate

MLflow logs batch training loss at every optimizer step. Each validation check also logs validation loss and interval-mean training loss. The notebook writes a snapshot, then copies it to Drive while training continues. It waits for the final copy before ending the run. Open the experiment in MLflow to plot the loss metrics against step.

In [ ]:
if train_model:
    if training_started:
        raise RuntimeError("Rerun the settings and initialization cells before starting training again.")

    with track_run(
        tracking_uri, MLFLOW_EXPERIMENT, {"preset": preset, "device": device.type, "precision": precision, **run_identity},
        run_id=mlflow_run_id if resume else None, backup=tracking_backup,
    ) as run:
        mlflow_run_id = run.id
        store = CheckpointStore(checkpoint_root, run.id)
        store.prepare()
        training_started = True
        run.set_tag("checkpoint_location", store.location)
        print(f"MLflow run ID: {run.id} | Drive checkpoints: {store.location}", flush=True)

        if resume:
            store.prune()
            for row in history:
                run.log(row, step=row["step"])
        else:
            print("Evaluating validation loss at step 0 before the first training batch...", flush=True)
            initial_loss = evaluate(model, val_data, config, device, precision, step=0)
            history = [{"step": 0, "tokens": 0, "val_loss": initial_loss, "val_targets": val_targets_per_check}]
            print(f"Step 0 | mean train loss N/A (no training batches yet) | validation loss {initial_loss:.{loss_decimals}f}", flush=True)
            run.log(history[0], step=0)

        with checkpoint_uploads(store, run) as save_checkpoint:
            interval_start = perf_counter()
            interval_start_targets = sampler.targets_through(start)
            interval_loss_sum = 0.0
            for step in range(start + 1, stop + 1):
                lr = config.lr_at(step)
                for group in optimizer.param_groups:
                    group["lr"] = lr
                optimizer.zero_grad(set_to_none=True)
                starts = sampler.starts(step)
                x, y = streams["train"].batch(starts, device)
                loss = loss_for(model, x, y, precision)
                tokens_seen = sampler.targets_through(step)
                batch_targets = tokens_seen - sampler.targets_through(step - 1)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss = loss.detach().item()
                if not math.isfinite(train_loss):
                    raise FloatingPointError(f"Non-finite training loss at step {step}.")
                print(f"Step {step:,} | train loss {train_loss:.{loss_decimals}f}", flush=True)
                interval_loss_sum += train_loss * batch_targets
                metrics = {"step": step, "train_loss": train_loss}
                if step % config.eval_every != 0 and step != stop:
                    run.log(metrics, step=step)
                    continue
                interval_targets = tokens_seen - interval_start_targets
                interval_mean = interval_loss_sum / interval_targets
                throughput = interval_targets / (perf_counter() - interval_start)
                val_loss = evaluate(model, val_data, config, device, precision, step=step)
                if not math.isfinite(val_loss):
                    raise FloatingPointError(f"Non-finite validation loss at step {step}.")
                print(
                    f"Step {step:,} | mean train loss {interval_mean:.{loss_decimals}f} | "
                    f"validation loss {val_loss:.{loss_decimals}f}",
                    flush=True,
                )
                metrics.update({
                    "tokens": tokens_seen,
                    "epoch": tokens_seen / streams["train"].target_count,
                    "train_loss_interval_mean": interval_mean,
                    "val_loss": val_loss,
                    "val_targets": val_targets_per_check,
                    "lr": lr,
                    "train_tokens_per_second": round(throughput),
                })
                run.log(metrics, step=step)
                history.append(metrics)
                print(f"Saving snapshot at step {step:,} for background upload...", flush=True)
                save_checkpoint(
                    {
                        "version": 1,
                        **run_identity,
                        "step": step,
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "tokenizer_json": tokenizer_json,
                        "metrics": history,
                    },
                )
                interval_loss_sum = 0.0
                interval_start_targets = tokens_seen
                interval_start = perf_counter()
    status = "Paused" if stop < config.steps else "Finished"
    print(
        f"{status} at step {stop:,} | final train loss {train_loss:.{loss_decimals}f} | "
        f"final validation loss {val_loss:.{loss_decimals}f}",
        flush=True,
    )
    print(f"Drive checkpoint saved for MLflow run {mlflow_run_id}", flush=True)

## Generate from a Drive checkpoint

After training, use the current run's latest checkpoint. For CPU inference in a new session, set **train_model = False**, **mlflow_run_id** to the saved ID, and use the same Drive checkpoint root. The generation cell reads directly from Drive.

In [ ]:
prompt = "Once upon a time, there was a little"
max_new_tokens = 80

checkpoint_alias = "latest"  # Use "best" for the lowest-loss retained model, or "step-N" for a retained step.

with download_checkpoint(CheckpointStore(checkpoint_root, mlflow_run_id), alias=checkpoint_alias) as checkpoint_path:
    completion = complete(checkpoint_path, prompt, max_new_tokens, torch.device(inference_device))
print(prompt + completion)